# 10 Final Report Generation

This notebook generates the final CV-job matching report.

It combines outputs from previous notebooks:

- CV quality analysis
- structured CV extraction
- structured job posting extraction
- hybrid matching and scoring
- recommendation generation

The goal of this notebook is not to calculate new scores or generate new recommendations.  
Instead, it organizes all previous results into one final structured output.

## 1. Imports and Settings

In [42]:
import json

from pathlib import Path
from datetime import datetime

In [43]:
CV_QUALITY_DIR = Path("../outputs/cv_quality")
CV_EXTRACTION_DIR = Path("../outputs/cv_extraction")
JOB_EXTRACTION_DIR = Path("../outputs/job_extraction")
MATCHING_DIR = Path("../outputs/matching")
RECOMMENDATIONS_DIR = Path("../outputs/recommendations")
FINAL_REPORT_DIR = Path("../outputs/final_report")

In [44]:
cv_quality_path = CV_QUALITY_DIR / "cv_quality_analysis.json"
structured_cv_path = CV_EXTRACTION_DIR / "structured_cv.json"
structured_job_path = JOB_EXTRACTION_DIR / "structured_job.json"
matching_result_path = MATCHING_DIR / "matching_result.json"
recommendations_path = RECOMMENDATIONS_DIR / "recommendations.json"

final_report_json_path = FINAL_REPORT_DIR / "final_report.json"
final_report_markdown_path = FINAL_REPORT_DIR / "final_report.md"

## 2. Load Input Data

In [45]:
with open(cv_quality_path, "r", encoding="utf-8") as file:
    cv_quality_analysis = json.load(file)

In [46]:
with open(structured_cv_path, "r", encoding="utf-8") as file:
    structured_cv = json.load(file)

In [47]:

with open(structured_job_path, "r", encoding="utf-8") as file:
    structured_job = json.load(file)

In [48]:
with open(matching_result_path, "r", encoding="utf-8") as file:
    matching_result = json.load(file)

In [49]:
with open(recommendations_path, "r", encoding="utf-8") as file:
    recommendations_output = json.load(file)

## 3. Define Helper Functions

In [50]:
def get_nested_value(data, keys, default=None):
    current_value = data

    for key in keys:
        if not isinstance(current_value, dict):
            return default

        current_value = current_value.get(key)

        if current_value is None:
            return default

    return current_value

In [51]:
def create_markdown_list(items, empty_message="- No items available."):
    if not items:
        return empty_message

    lines = []

    for item in items:
        if item is None:
            continue

        if isinstance(item, dict):
            item_text = item.get("job_requirement") or item.get("title") or item.get("skill") or str(item)
        else:
            item_text = str(item)

        if item_text.strip():
            lines.append(f"- {item_text}")

    if not lines:
        return empty_message

    return "\n".join(lines)

In [52]:
def create_matched_items_list(items, empty_message="- No matched items available."):
    if not items:
        return empty_message

    lines = []

    for item in items:
        if not isinstance(item, dict):
            lines.append(f"- {item}")
            continue

        job_requirement = item.get("job_requirement")
        cv_evidence = item.get("cv_evidence")
        match_type = item.get("match_type")

        line = f"- {job_requirement}"

        if cv_evidence:
            line += f" | CV evidence: {cv_evidence}"

        if match_type:
            line += f" | Match type: {match_type}"

        lines.append(line)

    return "\n".join(lines)

In [53]:
def create_recommendation_section(items, empty_message="- No recommendations available."):
    if not items:
        return empty_message

    lines = []

    for index, item in enumerate(items, start=1):
        title = item.get("title", f"Recommendation {index}")
        recommendation = item.get("recommendation", "")
        reason = item.get("reason", "")
        evidence = item.get("evidence", "")
        priority = item.get("priority", "")

        lines.append(f"### {index}. {title}")
        lines.append("")
        lines.append(f"- Priority: {priority}")
        lines.append(f"- Recommendation: {recommendation}")
        lines.append(f"- Reason: {reason}")
        lines.append(f"- Evidence: {evidence}")
        lines.append("")

    return "\n".join(lines)

In [54]:
def create_skill_recommendation_section(items, empty_message="- No skill recommendations available."):
    if not items:
        return empty_message

    lines = []

    for index, item in enumerate(items, start=1):
        skill = item.get("skill", f"Skill {index}")
        current_status = item.get("current_status", "")
        development_recommendation = item.get("development_recommendation", "")
        cv_update_recommendation = item.get("cv_update_recommendation", "")
        priority = item.get("priority", "")

        lines.append(f"### {index}. {skill}")
        lines.append("")
        lines.append(f"- Priority: {priority}")
        lines.append(f"- Current status: {current_status}")
        lines.append(f"- Development recommendation: {development_recommendation}")
        lines.append(f"- CV update recommendation: {cv_update_recommendation}")
        lines.append("")

    return "\n".join(lines)


In [55]:
def create_priority_actions_section(items, empty_message="- No priority actions available."):
    if not items:
        return empty_message

    lines = []

    for index, item in enumerate(items, start=1):
        action = item.get("action", "")
        expected_impact = item.get("expected_impact", "")
        priority = item.get("priority", "")

        lines.append(f"{index}. {action}")
        lines.append(f"   - Priority: {priority}")
        lines.append(f"   - Expected impact: {expected_impact}")

    return "\n".join(lines)

## 4. Extract Main Report Sections

In [56]:
cv_quality_scores = cv_quality_analysis.get("scores", {})

In [57]:
matching_final_result = matching_result.get("final_result", {})
matching_score_breakdown = matching_result.get("score_breakdown", {})
matched_items = matching_result.get("matched_items", {})
missing_items = matching_result.get("missing_items", {})
semantic_analysis = matching_result.get("semantic_analysis", {})

In [58]:
recommendations = recommendations_output.get("recommendations", recommendations_output)

In [59]:
candidate_information = {
    "candidate_name": structured_cv.get("candidate_name"),
    "email": structured_cv.get("email"),
    "phone": structured_cv.get("phone"),
    "location": structured_cv.get("location"),
    "linkedin": structured_cv.get("linkedin"),
    "github": structured_cv.get("github"),
    "portfolio": structured_cv.get("portfolio")
}

In [60]:
job_information = {
    "job_title": structured_job.get("job_title"),
    "company_name": structured_job.get("company_name"),
    "job_category": structured_job.get("job_category"),
    "location": structured_job.get("location"),
    "work_mode": structured_job.get("work_mode"),
    "employment_type": structured_job.get("employment_type")
}

## 5. Prepare CV Quality Summary

In [61]:
cv_quality_score_weights = {
    "structure_and_readability_score": 0.15,
    "completeness_score": 0.15,
    "technical_skills_clarity_score": 0.20,
    "experience_description_score": 0.20,
    "projects_description_score": 0.15,
    "measurable_results_score": 0.10,
    "it_relevance_score": 0.05
}

In [62]:
def define_cv_quality_category(score):
    if score is None:
        return None

    if score >= 85:
        return "Strong CV"
    elif score >= 70:
        return "Good CV"
    elif score >= 50:
        return "Basic CV"
    else:
        return "Weak CV"

In [63]:
def get_or_calculate_cv_quality_score(cv_quality_analysis, score_weights):
    possible_score_keys = [
        "final_cv_quality_score",
        "overall_cv_quality_score",
        "weighted_cv_quality_score",
        "cv_quality_score"
    ]

    for key in possible_score_keys:
        value = cv_quality_analysis.get(key)

        if isinstance(value, (int, float)):
            return round(value, 2)

    scores = cv_quality_analysis.get("scores", {})

    if not isinstance(scores, dict):
        return None

    total_score = 0

    for score_name, weight in score_weights.items():
        score_value = scores.get(score_name)

        if score_value is None:
            return None

        total_score += score_value * weight

    return round(total_score, 2)

In [64]:
cv_quality_final_score = get_or_calculate_cv_quality_score(
    cv_quality_analysis,
    cv_quality_score_weights
)

In [65]:
cv_quality_category = (
    cv_quality_analysis.get("cv_quality_category")
    or cv_quality_analysis.get("quality_category")
    or define_cv_quality_category(cv_quality_final_score)
)

In [66]:
cv_quality_summary = {
    "final_cv_quality_score": cv_quality_final_score,
    "cv_quality_category": cv_quality_category,
    "overall_summary": cv_quality_analysis.get("overall_summary"),
    "scores": cv_quality_scores,
    "strengths": cv_quality_analysis.get("strengths", []),
    "weaknesses": cv_quality_analysis.get("weaknesses", []),
    "missing_or_unclear_sections": cv_quality_analysis.get("missing_or_unclear_sections", []),
    "cv_improvement_recommendations": cv_quality_analysis.get("cv_improvement_recommendations", []),
    "evidence_notes": cv_quality_analysis.get("evidence_notes", [])
}

## 6. Prepare Matching Summary

In [67]:
matching_summary = {
    "final_hybrid_score": matching_final_result.get("final_hybrid_score"),
    "rule_based_score": matching_final_result.get("rule_based_score"),
    "semantic_score": matching_final_result.get("semantic_score"),
    "match_category": matching_final_result.get("match_category"),
    "score_breakdown": matching_score_breakdown
}

## 7. Prepare Skills and Requirements Summary

In [68]:
skills_and_requirements_summary = {
    "matched_required_skills": matched_items.get("matched_required_skills", []),
    "missing_required_skills": missing_items.get("missing_required_skills", []),
    "matched_nice_to_have_skills": matched_items.get("matched_nice_to_have_skills", []),
    "missing_nice_to_have_skills": missing_items.get("missing_nice_to_have_skills", []),
    "matched_technology_skills": matched_items.get("matched_technology_skills", []),
    "missing_technology_skills": missing_items.get("missing_technology_skills", []),
    "matched_certifications": matched_items.get("matched_certifications", []),
    "missing_certifications": missing_items.get("missing_certifications", []),
    "matched_languages": matched_items.get("matched_languages", []),
    "missing_languages": missing_items.get("missing_languages", [])
}

## 8. Prepare Semantic Analysis Summary

In [69]:
semantic_analysis_summary = {
    "responsibilities_alignment_score": semantic_analysis.get("responsibilities_alignment_score"),
    "soft_skills_evidence_score": semantic_analysis.get("soft_skills_evidence_score"),
    "role_fit_summary": semantic_analysis.get("role_fit_summary"),
    "responsibilities_evidenced": semantic_analysis.get("responsibilities_evidenced", []),
    "responsibilities_not_evidenced": semantic_analysis.get("responsibilities_not_evidenced", []),
    "soft_skills_evidenced": semantic_analysis.get("soft_skills_evidenced", []),
    "soft_skills_not_clearly_evidenced": semantic_analysis.get("soft_skills_not_clearly_evidenced", []),
    "evidence_notes": semantic_analysis.get("evidence_notes", [])
}

## 9. Prepare Recommendation Summary

In [70]:
recommendation_summary = {
    "overall_recommendation_summary": recommendations.get("overall_recommendation_summary"),
    "cv_improvement_recommendations": recommendations.get("cv_improvement_recommendations", []),
    "missing_required_skills_recommendations": recommendations.get("missing_required_skills_recommendations", []),
    "technical_development_recommendations": recommendations.get("technical_development_recommendations", []),
    "project_recommendations": recommendations.get("project_recommendations", []),
    "soft_skills_recommendations": recommendations.get("soft_skills_recommendations", []),
    "priority_actions": recommendations.get("priority_actions", []),
    "evidence_based_notes": recommendations.get("evidence_based_notes", [])
}

## 10. Create Final Report Dictionary

In [71]:
final_report = {
    "metadata": {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "report_type": "final_cv_job_matching_report",
        "source_files": {
            "cv_quality_analysis": str(cv_quality_path),
            "structured_cv": str(structured_cv_path),
            "structured_job": str(structured_job_path),
            "matching_result": str(matching_result_path),
            "recommendations": str(recommendations_path)
        }
    },
    "candidate_information": candidate_information,
    "job_information": job_information,
    "cv_quality_summary": cv_quality_summary,
    "matching_summary": matching_summary,
    "skills_and_requirements_summary": skills_and_requirements_summary,
    "semantic_analysis_summary": semantic_analysis_summary,
    "recommendation_summary": recommendation_summary
}

## 11. Save Final Report as JSON

In [72]:
with open(final_report_json_path, "w", encoding="utf-8") as file:
    json.dump(final_report, file, indent=4, ensure_ascii=False)

## 12. Generate Final Markdown Report

In [73]:
final_markdown_report = f"""
# Final CV-Job Matching Report

## 1. Candidate Information

- Candidate name: {candidate_information.get("candidate_name")}
- Email: {candidate_information.get("email")}
- Phone: {candidate_information.get("phone")}
- Location: {candidate_information.get("location")}
- LinkedIn: {candidate_information.get("linkedin")}
- GitHub: {candidate_information.get("github")}
- Portfolio: {candidate_information.get("portfolio")}

## 2. Job Information

- Job title: {job_information.get("job_title")}
- Company: {job_information.get("company_name")}
- Job category: {job_information.get("job_category")}
- Location: {job_information.get("location")}
- Work mode: {job_information.get("work_mode")}
- Employment type: {job_information.get("employment_type")}

## 3. CV Quality Summary

- Final CV quality score: {cv_quality_summary.get("final_cv_quality_score")}/100
- CV quality category: {cv_quality_summary.get("cv_quality_category")}

### Overall CV Quality Summary

{cv_quality_summary.get("overall_summary")}

### CV Quality Score Breakdown

- Structure and readability: {cv_quality_scores.get("structure_and_readability_score")}/100
- Completeness: {cv_quality_scores.get("completeness_score")}/100
- Technical skills clarity: {cv_quality_scores.get("technical_skills_clarity_score")}/100
- Experience description: {cv_quality_scores.get("experience_description_score")}/100
- Projects description: {cv_quality_scores.get("projects_description_score")}/100
- Measurable results: {cv_quality_scores.get("measurable_results_score")}/100
- IT relevance: {cv_quality_scores.get("it_relevance_score")}/100

### CV Strengths

{create_markdown_list(cv_quality_summary.get("strengths", []), empty_message="- No strengths listed.")}

### CV Weaknesses

{create_markdown_list(cv_quality_summary.get("weaknesses", []), empty_message="- No weaknesses listed.")}

### Missing or Unclear CV Sections

{create_markdown_list(cv_quality_summary.get("missing_or_unclear_sections", []), empty_message="- No missing or unclear sections listed.")}

## 4. Final Matching Summary

- Final hybrid match score: {matching_summary.get("final_hybrid_score")}/100
- Rule-based score: {matching_summary.get("rule_based_score")}/100
- LLM semantic score: {matching_summary.get("semantic_score")}/100
- Match category: {matching_summary.get("match_category")}

### Score Breakdown

- Required skills score: {matching_score_breakdown.get("required_skills_score")}/100
- Technology score: {matching_score_breakdown.get("technology_score")}/100
- Experience score: {matching_score_breakdown.get("experience_score")}/100
- Education score: {matching_score_breakdown.get("education_score")}/100
- Nice-to-have score: {matching_score_breakdown.get("nice_to_have_score")}/100
- Certification score: {matching_score_breakdown.get("certification_score")}/100
- Language score: {matching_score_breakdown.get("language_score")}/100

## 5. Rule-Based Matching Details

### Matched Required Skills

{create_matched_items_list(skills_and_requirements_summary.get("matched_required_skills", []), empty_message="- No matched required skills.")}

### Missing Required Skills

{create_markdown_list(skills_and_requirements_summary.get("missing_required_skills", []), empty_message="- No missing required skills.")}

### Matched Technology Skills

{create_matched_items_list(skills_and_requirements_summary.get("matched_technology_skills", []), empty_message="- No matched technology skills.")}

### Missing Technology Skills

{create_markdown_list(skills_and_requirements_summary.get("missing_technology_skills", []), empty_message="- No missing technology skills.")}

### Matched Nice-to-Have Skills

{create_matched_items_list(skills_and_requirements_summary.get("matched_nice_to_have_skills", []), empty_message="- No matched nice-to-have skills.")}

### Missing Nice-to-Have Skills

{create_markdown_list(skills_and_requirements_summary.get("missing_nice_to_have_skills", []), empty_message="- No missing nice-to-have skills.")}

## 6. LLM Semantic Analysis

### Role Fit Summary

{semantic_analysis_summary.get("role_fit_summary")}

### Responsibilities Alignment Score

{semantic_analysis_summary.get("responsibilities_alignment_score")}/100

### Soft Skills Evidence Score

{semantic_analysis_summary.get("soft_skills_evidence_score")}/100

### Responsibilities Evidenced in CV

{create_markdown_list(semantic_analysis_summary.get("responsibilities_evidenced", []), empty_message="- No responsibilities clearly evidenced.")}

### Responsibilities Not Clearly Evidenced

{create_markdown_list(semantic_analysis_summary.get("responsibilities_not_evidenced", []), empty_message="- No responsibility gaps listed.")}

### Soft Skills Evidenced in CV

{create_markdown_list(semantic_analysis_summary.get("soft_skills_evidenced", []), empty_message="- No soft skills clearly evidenced.")}

### Soft Skills Not Clearly Evidenced in CV

{create_markdown_list(semantic_analysis_summary.get("soft_skills_not_clearly_evidenced", []), empty_message="- No soft skill evidence gaps listed.")}

### Semantic Evidence Notes

{create_markdown_list(semantic_analysis_summary.get("evidence_notes", []), empty_message="- No semantic evidence notes listed.")}

## 7. Recommendation Summary

### Overall Recommendation Summary

{recommendation_summary.get("overall_recommendation_summary")}

## 8. CV Improvement Recommendations

{create_recommendation_section(recommendation_summary.get("cv_improvement_recommendations", []), empty_message="- No CV improvement recommendations.")}

## 9. Missing Required Skills Recommendations

{create_skill_recommendation_section(recommendation_summary.get("missing_required_skills_recommendations", []), empty_message="- No missing required skills recommendations.")}

## 10. Technical Development Recommendations

{create_skill_recommendation_section(recommendation_summary.get("technical_development_recommendations", []), empty_message="- No technical development recommendations.")}

## 11. Project Recommendations

{create_recommendation_section(recommendation_summary.get("project_recommendations", []), empty_message="- No project recommendations.")}

## 12. Soft Skills Recommendations

{create_recommendation_section(recommendation_summary.get("soft_skills_recommendations", []), empty_message="- No soft skills recommendations.")}

## 13. Priority Actions

{create_priority_actions_section(recommendation_summary.get("priority_actions", []), empty_message="- No priority actions.")}

## 14. Evidence-Based Notes and Limitations

{create_markdown_list(recommendation_summary.get("evidence_based_notes", []), empty_message="- No additional evidence-based notes.")}

## 15. Final Conclusion

This report combines CV quality analysis, structured CV extraction, structured job posting extraction, hybrid CV-job matching and recommendation generation.

The final result should be interpreted as an evidence-based support tool for CV improvement and job matching, not as a final hiring decision.

The system does not assume skills or experience that are not evidenced in the CV.
"""

## 13. Save Final Markdown Report

In [74]:
with open(final_report_markdown_path, "w", encoding="utf-8") as file:
    file.write(final_markdown_report)